# Notebook 2: Softmax, Attention, and Circuit Matrices

**Reading schedule:**
- **Before Ex 1–3:** nothing new — you know this from your blog.
- **Before Ex 5–6:** read Elhage 2021 §1–3 — [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html). Focus on:
  - The residual stream view
  - W_QK = W_Q @ W_K.T and W_OV = W_V @ W_O — why these objects, not the factors
  - The full-attention formula as a sum of head contributions

**Convention (Elhage 2021):**
```
W_QK = W_Q @ W_K.T      # (d_model, d_model)
W_OV = W_V @ W_O        # (d_model, d_model)  — W_O is (d_v, d_model) per head
```


In [1]:
import numpy as np

def softmax(z, axis=-1):
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def check(name, got, expected=None, cond=None, rtol=1e-4, atol=1e-6):
    try:
        if cond is not None:
            passed = bool(cond(got))
        elif expected is not None:
            passed = np.allclose(np.array(got, dtype=float),
                                 np.array(expected, dtype=float), rtol=rtol, atol=atol)
        else:
            passed = True
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed and expected is not None:
            g, e = np.array(got), np.array(expected)
            print(f"         got shape {g.shape}, expected shape {e.shape}")
            if g.shape == e.shape and g.size <= 16:
                print(f"         got:      {g.ravel()}")
                print(f"         expected: {e.ravel()}")
        return passed
    except Exception as exc:
        print(f"  [ERROR] {name}: {type(exc).__name__}: {exc}")
        return False

print("Helper loaded. softmax and check() are available.")


Helper loaded. softmax and check() are available.


## Ex 1: Numerically stable softmax

In [2]:
def softmax(z, axis=-1):
    """Stable softmax. z: any shape. Normalizes along `axis`."""
    z-=z.max(axis=axis, keepdims=True)
    e=np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


In [5]:
def softmax(z, axis=-1):
    z -= np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

In [6]:
print("Ex 1: softmax")
z = np.array([1., 2., 3.])
s = softmax(z)
check("sums to 1", s.sum(), 1.0)
check("monotone", cond=lambda x: np.all(np.diff(x) > 0), got=s)
# GOTCHA: overflow on large values
s_big = softmax(np.array([1000., 1001., 1002.]))
check("stable on large z (finite)", cond=lambda x: np.all(np.isfinite(x)), got=s_big)
check("large z sums to 1", s_big.sum(), 1.0)
Z = np.random.randn(4, 6)
check("2D rows sum to 1", softmax(Z, axis=-1).sum(axis=-1), np.ones(4))
check("2D cols sum to 1 (axis=0)", softmax(Z, axis=0).sum(axis=0), np.ones(6))


Ex 1: softmax
  [PASS] sums to 1
  [PASS] monotone
  [PASS] stable on large z (finite)
  [PASS] large z sums to 1
  [PASS] 2D rows sum to 1
  [PASS] 2D cols sum to 1 (axis=0)


True

## Ex 2: log_softmax + cross-entropy

In [13]:
def log_softmax(z, axis=-1):
    """Stable log_softmax. Do NOT compute softmax() first then log() — that loses precision."""
    z -= z.max(axis=axis, keepdims = True)
    return z - np.log( np.exp(z).sum(axis = axis, keepdims=True) )

def cross_entropy(logits, targets):
    """
    logits: (B, C) float. targets: (B,) int indices.
    Returns scalar mean cross-entropy loss.
    """
    B=logits.shape[0]
    lsm = log_softmax(logits, axis=-1)
    return -lsm[np.arange(B),targets].mean()


In [14]:
# targets

In [15]:
print("Ex 2: log_softmax")
z = np.array([1., 2., 3.])
ls = log_softmax(z)
check("equals log(softmax)", ls, np.log(softmax(z)), rtol=1e-5)
check("stable on large z", cond=lambda x: np.all(np.isfinite(x)),
      got=log_softmax(np.array([1000., 1001., 1002.])))
print()
print("Ex 2: cross_entropy")
logits = np.array([[5., 0., 0.], 
                   [0., 5., 0.], 
                   [0., 0., 5.]] )
targets = np.array([0, 1, 2])
loss = cross_entropy(logits, targets)
check("low loss on correct preds", cond=lambda l: l < 0.1, got=loss)
check("uniform → log(C)", cross_entropy(np.zeros((4,3)), np.array([0,1,2,0])), np.log(3), rtol=1e-4)
check("scalar output", cond=lambda l: np.array(l).ndim == 0, got=loss)


Ex 2: log_softmax
  [PASS] equals log(softmax)
  [PASS] stable on large z

Ex 2: cross_entropy
  [PASS] low loss on correct preds
  [PASS] uniform → log(C)
  [PASS] scalar output


True

In [16]:
cross_entropy(logits, targets)

0.013385901721448773

## Ex 3: Softmax Jacobian

In [37]:
def softmax_jacobian(z, axis=-1):
    """
    J[i,j] = pi_i * (delta_ij - pi_j)
    z: (n,) 1D. Returns (n, n).
    """
    pi_ = softmax(z )
    outer_ = np.einsum("i,j->ij", pi_,pi_)
    print(outer_.shape)
    return np.diag(pi_) - outer_

In [38]:
print("Ex 3: softmax_jacobian")
z = np.array([1., 2., 0.5, -1.])
pi = softmax(z)
J = softmax_jacobian(z)
check("shape (4,4)", cond=lambda x: x.shape == (4,4), got=J)
check("diagonal: pi*(1-pi)", J.diagonal(), pi*(1-pi))
check("off-diag [0,1]: -pi0*pi1", J[0,1], -pi[0]*pi[1])
check("rows sum to 0", J.sum(axis=1), np.zeros(4), atol=1e-10)
# Verify with finite differences
eps = 1e-5
J_fd = np.zeros((4,4))
for j in range(4):
    zp, zm = z.copy(), z.copy(); zp[j] += eps; zm[j] -= eps
    J_fd[:, j] = (softmax(zp) - softmax(zm)) / (2*eps)
check("matches finite differences", J, J_fd, rtol=1e-4)


Ex 3: softmax_jacobian
(4, 4)
  [PASS] shape (4,4)
  [PASS] diagonal: pi*(1-pi)
  [PASS] off-diag [0,1]: -pi0*pi1
  [PASS] rows sum to 0
  [PASS] matches finite differences


True

## Ex 4: Single-head causal attention

Apply causal mask **before** softmax, not after.
Mask: set score[i, j] = −1e9 for all j > i (future tokens).


In [93]:
def attention(X, W_Q, W_K, W_V, causal=True):
    """
    X: (T, d_model). W_Q/K: (d_model, d_k). W_V: (d_model, d_v).
    Returns: output (T, d_v), weights (T, T).
    """
#     scores = X @ W_Q @ W_K.T @ X.T / np.sqrt(W_Q.shape[-1])
    scores = np.einsum("im,ma,na,jn->ij", X, W_Q, W_K, X) / np.sqrt(W_Q.shape[-1])
    if causal:
        T = X.shape[0]
        mask_idx = np.triu(     np.ones( (T,T), dtype=bool )     , k=1 )
        scores[mask_idx] = -1e9
        
        
    print(scores.shape)    
    A_ = softmax( scores, axis = -1)
    values = X @ W_V
    return A_ @ values , A_

#     scores = X @ W_Q @ W_K.T @ X.T / np.sqrt(W_Q.shape[-1])
#     T = X.shape[0]
#     if causal:
#         mask = np.triu(    np.ones( (T, T), dtype=bool )   ,   k=1  )  # upper triangle, above diagonal
#         scores[mask]=-1e9
    
#     weights = softmax(scores, axis=-1)
#     values = X @ W_V
#     return weights @ values, weights
    


In [94]:
print("Ex 4: attention")
T, dm, dk, dv = 5, 8, 4, 6
X = np.random.randn(T, dm)
W_Q = np.random.randn(dm, dk); W_K = np.random.randn(dm, dk); W_V = np.random.randn(dm, dv)
out, w = attention(X, W_Q, W_K, W_V, causal=True)
check("output shape (T,dv)", cond=lambda x: x.shape == (T,dv), got=out)
check("weights shape (T,T)", cond=lambda x: x.shape == (T,T), got=w)
check("weights sum to 1", w.sum(axis=-1), np.ones(T))
check("causal: upper triangle ~0",
      cond=lambda x: np.allclose(np.triu(x, k=1), 0, atol=1e-6), got=w)
check("pos 0 attends only to itself", cond=lambda x: np.isclose(x[0,0], 1.0), got=w)
out_nc, w_nc = attention(X, W_Q, W_K, W_V, causal=False)
check("no mask: upper tri NOT all zero",
      cond=lambda x: not np.allclose(np.triu(x, k=1), 0), got=w_nc)
# GOTCHA: divided by sqrt(d_k)?
Q = X @ W_Q; K = X @ W_K; V = X @ W_V
mask = np.triu(np.ones((T,T)), k=1)
w_manual = softmax(Q @ K.T / np.sqrt(dk) - 1e9*mask, axis=-1)
check("scaling is sqrt(d_k)", w, w_manual)


Ex 4: attention
(5, 5)
  [PASS] output shape (T,dv)
  [PASS] weights shape (T,T)
  [PASS] weights sum to 1
  [PASS] causal: upper triangle ~0
  [PASS] pos 0 attends only to itself
(5, 5)
  [PASS] no mask: upper tri NOT all zero
  [PASS] scaling is sqrt(d_k)


True

In [ ]:
  scores = X @ W_Q @ W_K.T @ X.T / np.sqrt(W_Q.shape[-1])
    T = X.shape[0]
    if causal:
        mask = np.triu(    np.ones( (T, T), dtype=bool )   ,   k=1  )  # upper triangle, above diagonal
        scores[mask]=-1e9
    
    weights = softmax(scores, axis=-1)
    values = X @ W_V
    return weights @ values, weights

## Ex 5: Multi-head attention

In [99]:
W_Qs[1,:,:].shape

(16, 4)

In [106]:
def multi_head_attention(X, W_Qs, W_Ks, W_Vs, W_O, causal=True):
    """
    X: (T, d_model). W_Qs/Ks/Vs: (H, d_model, d_k or d_v). W_O: (H*d_v, d_model).
    Returns: output (T, d_model), list of H weight matrices each (T,T).
    """
    d_k = W_Ks.shape[-1]
    H = W_Ks.shape[0]
    T= X.shape[0]
    d_model = W_O.shape[-1]
    d_v = W_Vs.shape[-1]
    
#     print(W_Q.shape)
    
    scores = np.einsum("im,hma,hna,jn->hij", X,W_Qs, W_Ks, X) / np.sqrt(d_k) # (H, T, T)
    if causal == True:
        mask = np.triu(     np.ones( (H,T,T), dtype=bool )  ,  k=1   )
        scores[mask] = -1e9
        
    A = softmax(scores, axis=-1) # H, T, T
    values = np.einsum("jm,hma->hja", X, W_Vs) # H, T, d_v
    
    head_output = np.einsum("hij,hja->iha", A, values) # i, H, d_v
    output = head_output.reshape(T, -1) @ W_O
    
    return output, [ A[h] for h in range(H) ]

In [107]:
print("Ex 5: multi_head_attention")
T, dm, H, dk, dv = 6, 16, 4, 4, 4
X = np.random.randn(T, dm)
W_Qs = np.random.randn(H, dm, dk); W_Ks = np.random.randn(H, dm, dk)
W_Vs = np.random.randn(H, dm, dv); W_O = np.random.randn(H*dv, dm)
out, ws = multi_head_attention(X, W_Qs, W_Ks, W_Vs, W_O)
check("output shape (T,d_model)", cond=lambda x: x.shape == (T,dm), got=out)
check("H weight matrices", cond=lambda x: len(x) == H, got=ws)
check("each (T,T)", cond=lambda x: all(w.shape == (T,T) for w in x), got=ws)
# Head 0 weights must match single-head
_, w0 = attention(X, W_Qs[0], W_Ks[0], W_Vs[0])
check("head 0 weights match single-head", ws[0], w0)


Ex 5: multi_head_attention
  [PASS] output shape (T,d_model)
  [PASS] H weight matrices
  [PASS] each (T,T)
(6, 6)
  [PASS] head 0 weights match single-head


True

## Ex 6: Circuit matrices W_QK and W_OV

From Elhage 2021:
- **W_QK = W_Q @ W_K.T** — (d_model, d_model)
  Attention score i↔j: `x_i @ W_QK @ x_j / √d_k`
- **W_OV = W_V @ W_O_head** — (d_model, d_model)
  Output contribution from position i: `attention_weight * (x_i @ W_OV)`

GOTCHA: W_QK = W_Q @ W_K.T, **not** W_Q.T @ W_K.


In [110]:
def circuit_matrices(W_Q, W_K, W_V, W_O_head):
    """
    W_Q/K: (d_model, d_k). W_V: (d_model, d_v). W_O_head: (d_v, d_model).
    Returns W_QK (d_model, d_model), W_OV (d_model, d_model).
    """
    W_QK = np.einsum("ma,na->mn", W_Q, W_K)
    W_OV = np.einsum("ma,an->mn", W_V, W_O_head)
    return W_QK , W_OV


In [111]:
print("Ex 6: circuit_matrices")
dm, dk, dv = 8, 4, 4
W_Q = np.random.randn(dm, dk); W_K = np.random.randn(dm, dk)
W_V = np.random.randn(dm, dv); W_O_h = np.random.randn(dv, dm)
W_QK, W_OV = circuit_matrices(W_Q, W_K, W_V, W_O_h)
check("W_QK shape (dm,dm)", cond=lambda x: x.shape == (dm,dm), got=W_QK)
check("W_OV shape (dm,dm)", cond=lambda x: x.shape == (dm,dm), got=W_OV)
# W_QK should reproduce attention scores
T = 5; X = np.random.randn(T, dm)
direct = (X @ W_Q) @ (X @ W_K).T / np.sqrt(dk)
circuit = (X @ W_QK @ X.T) / np.sqrt(dk)
check("W_QK reproduces scores", direct, circuit)
# W_OV should reproduce output direction
x = np.random.randn(dm)
check("W_OV correctly ordered", (x @ W_V) @ W_O_h, x @ W_OV)


Ex 6: circuit_matrices
  [PASS] W_QK shape (dm,dm)
  [PASS] W_OV shape (dm,dm)
  [PASS] W_QK reproduces scores
  [PASS] W_OV correctly ordered


True